# LeetCode #200 — Number of Islands: đếm thành phần liên thông trên lưới (DFS / BFS / Union-Find)

**Mức độ:** Medium · **Chủ đề:** Array, DFS, BFS, Union Find, Matrix

Skeleton trên LeetCode:

```python
class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
```

Mục tiêu của notebook:

1. Xem lưới như **đồ thị ẩn**: mỗi ô đất là đỉnh, nối với các ô đất **kề 4 hướng**.
2. Số đảo = số **thành phần liên thông** = số lần phải bắt đầu một lượt **tô loang** mới.
3. Ba cách: DFS (đệ quy và ngăn xếp), BFS, **Union-Find** (hợp nhất tập).
4. Bẫy thực tế: đệ quy quá sâu trên lưới 300 × 300; đánh dấu đã thăm đúng lúc.

# 1. Bài toán là gì?

Cho lưới `m × n` gồm `'1'` (đất) và `'0'` (nước), đếm số **đảo**. Một đảo là nhóm ô đất nối với nhau theo chiều **ngang hoặc dọc** (không tính chéo) và bao quanh bởi nước. Coi như ngoài lưới là nước.

**Constraints (theo đề):**

- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 300`
- `grid[i][j]` là `'0'` hoặc `'1'`.

---

## Ví dụ

```text
11110
11010        ->  1
11000
00000

11000
11000        ->  3
00100
00011
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ma trận (mảng hai chiều) trong Python
2. Duyệt theo chiều rộng (BFS — Breadth-First Search)
3. Duyệt theo chiều sâu (DFS — Depth-First Search)
4. Union-Find (Disjoint Set Union)

Cuối mục có phần **2.5 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ma trận (mảng hai chiều) trong Python

### Tạo và truy cập

Ma trận `m × n` là list gồm `m` hàng, mỗi hàng là list `n` phần tử. Ô hàng `i`, cột `j` là `M[i][j]`; số hàng `len(M)`, số cột `len(M[0])`.

```text
        j=0 j=1 j=2
i=0   [  1,  2,  3 ]
i=1   [  4,  5,  6 ]
```

In [1]:
M_mx = [[1, 2, 3],
        [4, 5, 6]]
print("M[1][2] =", M_mx[1][2], "| số hàng:", len(M_mx), "| số cột:", len(M_mx[0]))

dung_mx = [[0] * 3 for _ in range(2)]    # ĐÚNG: mỗi hàng là list riêng
sai_mx = [[0] * 3] * 2                   # SAI: 2 hàng là cùng một list
dung_mx[0][0] = sai_mx[0][0] = 7
print("tạo bằng for:", dung_mx, "| nhân list:", sai_mx)

M[1][2] = 6 | số hàng: 2 | số cột: 3
tạo bằng for: [[7, 0, 0], [0, 0, 0]] | nhân list: [[7, 0, 0], [7, 0, 0]]


### Duyệt và các phép biến đổi cơ bản

- Duyệt theo hàng: `for i in range(m): for j in range(n)`.
- **Chuyển vị** (transpose): `T[j][i] = M[i][j]` — hàng thành cột. Trong Python: `list(zip(*M))`.
- **Lật ngang** một hàng: `row[::-1]`; lật dọc: đảo thứ tự hàng `M[::-1]`.
- Xoay 90° theo chiều kim đồng hồ = chuyển vị rồi lật từng hàng.

In [2]:
def in_mt(A):
    for row in A:
        print("   ", " ".join(f"{v:>2}" for v in row))

A_mx = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print("gốc:"); in_mt(A_mx)
T_mx = [list(r) for r in zip(*A_mx)]
print("chuyển vị:"); in_mt(T_mx)
print("xoay 90° phải (chuyển vị + lật hàng):"); in_mt([r[::-1] for r in T_mx])

gốc:
     1  2  3
     4  5  6
     7  8  9
chuyển vị:
     1  4  7
     2  5  8
     3  6  9
xoay 90° phải (chuyển vị + lật hàng):
     7  4  1
     8  5  2
     9  6  3


### Ô kề và mảng hướng

Bốn ô kề của `(r, c)`: lên `(r−1, c)`, xuống `(r+1, c)`, trái `(r, c−1)`, phải `(r, c+1)`. Viết thành **mảng hướng** để khỏi lặp code, và luôn **kiểm tra biên** `0 <= nr < m and 0 <= nc < n`.

In [3]:
HUONG = [(-1, 0), (1, 0), (0, -1), (0, 1)]       # lên, xuống, trái, phải
def o_ke(r, c, m, n):
    return [(r + dr, c + dc) for dr, dc in HUONG if 0 <= r + dr < m and 0 <= c + dc < n]
print("ô kề của (0, 0) trong lưới 3×3:", o_ke(0, 0, 3, 3))
print("ô kề của (1, 1):", o_ke(1, 1, 3, 3))

ô kề của (0, 0) trong lưới 3×3: [(1, 0), (0, 1)]
ô kề của (1, 1): [(0, 1), (2, 1), (1, 0), (1, 2)]


### Đánh số ô thành một chỉ số và ngược lại

Ma trận `m × n` "trải phẳng" theo hàng: ô `(i, j)` ↔ chỉ số `k = i * n + j`; ngược lại `i, j = divmod(k, n)`. Hữu ích để tìm nhị phân trên ma trận hoặc lưu ô vào set.

In [4]:
n_cols = 4
for k in [0, 5, 11]:
    print(f"k = {k:>2} → (i, j) = {divmod(k, n_cols)} → k = {divmod(k, n_cols)[0] * n_cols + divmod(k, n_cols)[1]}")

k =  0 → (i, j) = (0, 0) → k = 0
k =  5 → (i, j) = (1, 1) → k = 5
k = 11 → (i, j) = (2, 3) → k = 11


Lỗi hay gặp: `[[0]*n]*m`; nhầm `m` (hàng) với `n` (cột) khi ma trận không vuông; quên kiểm tra biên; sửa ma trận trong lúc vẫn cần giá trị cũ để tính ô khác.

## 2.2 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [5]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [6]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [7]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E:

 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.3 Duyệt theo chiều sâu (DFS — Depth-First Search)

### DFS là gì?

DFS đi **sâu nhất có thể** theo một nhánh, tới khi hết đường thì **quay lui** về ngã rẽ gần nhất và thử nhánh khác — giống đi mê cung bằng cách luôn bám một bức tường.

```text
        A
       / \          DFS từ A (thử hàng xóm theo thứ tự):
      B   C         A → B → D (cụt, quay lui) → E → F → C
     / \   \
    D   E - F
```

Công cụ: **đệ quy** (dùng call stack) hoặc **ngăn xếp** tự quản lý.

### Khung DFS

```text
def dfs(u):
    visited.add(u)
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            dfs(v)
```

In [8]:
do_thi_dfs = {
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def dfs_de_quy(graph, u, visited, thu_tu, sau=0):
    visited.add(u)
    thu_tu.append(u)
    print("    " * sau + f"vào {u}")
    for v in graph[u]:
        if v not in visited:
            dfs_de_quy(graph, v, visited, thu_tu, sau + 1)
    print("    " * sau + f"rời {u}")

tt = []
dfs_de_quy(do_thi_dfs, "A", set(), tt)
print("Thứ tự DFS:", tt)

vào A
    vào B
        vào D
        rời D
        vào E
            vào F
                vào C
                rời C
            rời F
        rời E
    rời B
rời A
Thứ tự DFS: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS bằng ngăn xếp (không đệ quy)

Thay hàng đợi của BFS bằng **ngăn xếp**. Tránh được giới hạn độ sâu đệ quy khi dữ liệu lớn.

In [9]:
def dfs_ngan_xep(graph, start):
    st, visited, thu_tu = [start], set(), []
    while st:
        u = st.pop()
        if u in visited:
            continue
        visited.add(u)
        thu_tu.append(u)
        for v in reversed(graph[u]):        # đảo để hàng xóm đầu tiên được lấy ra trước
            if v not in visited:
                st.append(v)
    return thu_tu

print("DFS bằng ngăn xếp:", dfs_ngan_xep(do_thi_dfs, "A"))

DFS bằng ngăn xếp: ['A', 'B', 'D', 'E', 'F', 'C']


### DFS trên lưới: tô loang (flood fill) và đếm vùng

Lưới là đồ thị ẩn: mỗi ô nối với 4 ô kề. DFS từ một ô sẽ "loang" khắp vùng liên thông chứa nó. Đếm số lần phải bắt đầu DFS mới = số vùng.

In [10]:
def dem_dao(grid):
    g = [list(row) for row in grid]
    R, C = len(g), len(g[0])
    def loang(r, c):
        if r < 0 or r >= R or c < 0 or c >= C or g[r][c] != "1":
            return                              # ra ngoài / nước / đã thăm
        g[r][c] = "#"                           # đánh dấu đã thăm
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            loang(r + dr, c + dc)
    dem = 0
    for r in range(R):
        for c in range(C):
            if g[r][c] == "1":
                dem += 1
                loang(r, c)
    return dem, ["".join(row) for row in g]

print(dem_dao(["11000",
               "11010",
               "00011"]))

(2, ['##000', '##0#0', '000##'])


### DFS trên cây: mang thông tin dọc đường đi

Trên cây không cần `visited` (không có chu trình). DFS hay **truyền xuống** thông tin của đường đi từ gốc (tổng, đường đi hiện tại) và **trả lên** kết quả của cây con.

In [11]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_dfs = TreeNode(5, TreeNode(4, TreeNode(11, TreeNode(7), TreeNode(2))), TreeNode(8, TreeNode(13), TreeNode(4)))

def moi_duong_goc_la(node, duong, out):
    if node is None:
        return
    duong.append(node.val)                     # đi xuống: thêm nút
    if node.left is None and node.right is None:
        out.append((list(duong), sum(duong)))  # tới lá: ghi lại một bản sao
    moi_duong_goc_la(node.left, duong, out)
    moi_duong_goc_la(node.right, duong, out)
    duong.pop()                                # quay lên: bỏ nút

ket_qua = []
moi_duong_goc_la(cay_dfs, [], ket_qua)
for d, s in ket_qua:
    print(d, "tổng =", s)

[5, 4, 11, 7] tổng = 27
[5, 4, 11, 2] tổng = 22
[5, 8, 13] tổng = 26
[5, 8, 4] tổng = 17


### So sánh BFS và DFS

| | BFS | DFS |
|---|---|---|
| Cấu trúc | hàng đợi | ngăn xếp / đệ quy |
| Thứ tự | theo lớp (gần → xa) | theo nhánh (sâu trước) |
| Đường ngắn nhất (không trọng số) | **có** | không đảm bảo |
| Bộ nhớ | độ rộng lớn nhất | độ sâu lớn nhất |
| Hợp với | khoảng cách, theo tầng | liệt kê đường đi, vùng liên thông, quay lui |

Lỗi hay gặp: quên `visited` trên đồ thị có chu trình (lặp vô hạn); đệ quy quá sâu trên lưới lớn; ghi `out.append(duong)` thay vì bản sao `list(duong)` (về sau list bị sửa).

## 2.4 Union-Find (Disjoint Set Union)

### Dùng để làm gì?

Quản lý các phần tử chia thành những **nhóm rời nhau**, hỗ trợ hai thao tác:

- `find(x)`: x thuộc nhóm nào (trả về **đại diện** của nhóm).
- `union(a, b)`: gộp nhóm của `a` và nhóm của `b`.

Ứng dụng: đếm **thành phần liên thông**, phát hiện cạnh tạo chu trình trong đồ thị vô hướng, Kruskal…

### Biểu diễn: mỗi nhóm là một cây, gốc là đại diện

```text
parent: 0→0, 1→0, 2→1, 3→3       nhóm {0, 1, 2} gốc 0;  nhóm {3} gốc 3
        0        3
        |
        1
        |
        2
```

### Hai tối ưu quan trọng

- **Nén đường đi** (path compression): khi `find(x)`, cho mọi nút trên đường đi trỏ thẳng về gốc.
- **Hợp theo kích thước/hạng**: gắn cây **nhỏ** vào cây **lớn** để cây không cao.

Với cả hai, mỗi thao tác gần như `O(1)` (chính xác là `O(α(n))`, α tăng cực chậm).

In [12]:
class DSUDemo:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.groups = n

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:          # nén đường đi
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                       # đã cùng nhóm
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.groups -= 1
        return True

d = DSUDemo(6)
for a, b in [(0, 1), (1, 2), (3, 4), (2, 0)]:
    print(f"union({a}, {b}) → gộp mới? {d.union(a, b)}, số nhóm = {d.groups}")
print("cùng nhóm 0 và 2?", d.find(0) == d.find(2), "| 0 và 3?", d.find(0) == d.find(3))

union(0, 1) → gộp mới? True, số nhóm = 5
union(1, 2) → gộp mới? True, số nhóm = 4
union(3, 4) → gộp mới? True, số nhóm = 3
union(2, 0) → gộp mới? False, số nhóm = 3
cùng nhóm 0 và 2? True | 0 và 3? False


### Lỗi hay gặp

- So sánh `parent[a] == parent[b]` thay vì `find(a) == find(b)` (cha trực tiếp chưa chắc là gốc).
- Gán `parent[a] = b` thay vì `parent[find(a)] = find(b)` → nhóm bị tách sai.
- `find` đệ quy không nén trên chuỗi dài → vượt giới hạn đệ quy.

## 2.5 Áp dụng vào bài này

Lưới là một đồ thị ẩn: mỗi ô đất là một đỉnh, nối với các ô đất **kề cạnh** (mảng hướng 4 chiều). Số đảo = số **thành phần liên thông**: quét từng ô, gặp đất chưa thăm thì đếm 1 và **loang** (DFS hoặc BFS) đánh dấu cả hòn đảo. Cách thứ ba: **Union-Find** gộp các ô đất kề nhau và đếm số nhóm.

# 3. Nền tảng

In [13]:
from typing import List
from collections import deque
import random, sys, time

def to_grid(rows: List[str]) -> List[List[str]]:
    return [list(r) for r in rows]

def show(grid):
    for row in grid:
        print("   ", "".join(row))

G1 = ["11110", "11010", "11000", "00000"]
G2 = ["11000", "11000", "00100", "00011"]
show(to_grid(G2))

DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
print("ô kề của (1, 1):", [(1 + dr, 1 + dc) for dr, dc in DIRS])

    11000
    11000
    00100
    00011
ô kề của (1, 1): [(2, 1), (0, 1), (1, 2), (1, 0)]


Đầu vào là `list[list[str]]` — các phần tử là **chuỗi** `'1'`/`'0'`, không phải số `1`/`0` (so `grid[r][c] == 1` luôn sai).

# 4. Ý tưởng chính: quét lưới, gặp đất chưa thăm thì tô loang cả đảo

```text
count = 0
for mỗi ô (r, c):
    if ô là đất và chưa thăm:
        count += 1
        tô loang (DFS/BFS) từ (r, c), đánh dấu MỌI ô đất của đảo này là đã thăm
return count
```

Mỗi đảo được "phát hiện" đúng **một lần** — ở ô đầu tiên của nó trong thứ tự quét; mọi ô khác của đảo đã bị đánh dấu trước khi vòng quét tới chúng.

Cách đánh dấu gọn: **đổi** ô đất thành `'0'` (hoặc ký hiệu khác) — "nhấn chìm" đảo. Nếu không được sửa đầu vào thì dùng `set` đã thăm.

## Cách 1: DFS đệ quy

In [14]:
def num_islands_dfs(grid: List[List[str]]) -> int:
    m, n = len(grid), len(grid[0])
    def sink(r, c):
        if r < 0 or r >= m or c < 0 or c >= n or grid[r][c] != "1":
            return
        grid[r][c] = "0"                       # đánh dấu đã thăm
        for dr, dc in DIRS:
            sink(r + dr, c + dc)
    count = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == "1":
                count += 1
                sink(r, c)
    return count

print(num_islands_dfs(to_grid(G1)), num_islands_dfs(to_grid(G2)))

1

 3


## Cách 2: BFS (hàng đợi) — không lo độ sâu đệ quy

Đánh dấu ô **ngay khi đưa vào hàng đợi** để một ô không bị đưa vào nhiều lần.

In [15]:
def num_islands_bfs(grid: List[List[str]]) -> int:
    m, n = len(grid), len(grid[0])
    count = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] != "1":
                continue
            count += 1
            grid[r][c] = "0"
            q = deque([(r, c)])
            while q:
                x, y = q.popleft()
                for dr, dc in DIRS:
                    nx, ny = x + dr, y + dc
                    if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == "1":
                        grid[nx][ny] = "0"
                        q.append((nx, ny))
    return count

print(num_islands_bfs(to_grid(G1)), num_islands_bfs(to_grid(G2)))

1 3


# 5. Cách 3: Union-Find (hợp nhất tập — Disjoint Set Union)

**Union-Find** quản lý một họ các tập **rời nhau** với hai thao tác:

- `find(x)`: tìm **đại diện** (gốc) của tập chứa `x`.
- `union(a, b)`: hợp hai tập chứa `a` và `b` thành một.

Mỗi tập là một **cây**; `parent[x]` trỏ lên cha, gốc trỏ về chính nó. Hai kỹ thuật làm nó gần như `O(1)`:

- **Nén đường đi** (path compression): khi `find`, cho mọi nút trên đường trỏ thẳng lên gốc.
- **Hợp theo kích thước/hạng**: gắn cây nhỏ dưới cây lớn.

Cho bài này: mỗi ô đất là một tập riêng; với mỗi cặp ô đất kề nhau thì `union`. Số đảo = số tập còn lại (đếm bằng biến `components`, giảm 1 mỗi lần `union` thành công).

In [16]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.components = n

    def find(self, x):
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:            # nén đường đi
            self.parent[x], x = root, self.parent[x]
        return root

    def union(self, a, b) -> bool:
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra                      # cây nhỏ dưới cây lớn
        self.size[ra] += self.size[rb]
        self.components -= 1
        return True


d = DSU(5)
d.union(0, 1); d.union(3, 4); d.union(1, 4)
print("parent:", d.parent, "| số tập:", d.components, "| 0 và 3 cùng tập?", d.find(0) == d.find(3))

parent:

 [0, 0, 2, 0, 3] | số tập: 2 | 0 và 3 cùng tập? True


In [17]:
def num_islands_dsu(grid: List[List[str]]) -> int:
    m, n = len(grid), len(grid[0])
    land = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == "1"]
    dsu = DSU(m * n)
    dsu.components = len(land)                    # chỉ đếm ô đất
    for r, c in land:
        for dr, dc in [(1, 0), (0, 1)]:           # chỉ cần nối xuống và sang phải
            nr, nc = r + dr, c + dc
            if nr < m and nc < n and grid[nr][nc] == "1":
                dsu.union(r * n + c, nr * n + nc)
    return dsu.components

print(num_islands_dsu(to_grid(G1)), num_islands_dsu(to_grid(G2)))

1 3


# 6. Mô phỏng: ghi nhãn đảo (A, B, C, …) vào từng ô

In [18]:
def trace_islands(rows):
    g = to_grid(rows)
    m, n = len(g), len(g[0])
    label = 0
    for r in range(m):
        for c in range(n):
            if g[r][c] == "1":
                label += 1
                tag = chr(ord("A") + label - 1)          # nhãn A, B, C… (không dùng "1" vì trùng ký hiệu đất!)
                q = deque([(r, c)]); g[r][c] = tag
                while q:
                    x, y = q.popleft()
                    for dr, dc in DIRS:
                        nx, ny = x + dr, y + dc
                        if 0 <= nx < m and 0 <= ny < n and g[nx][ny] == "1":
                            g[nx][ny] = tag; q.append((nx, ny))
                print(f"phát hiện đảo {tag} tại ({r}, {c})")
    print("lưới sau khi gán nhãn (· là nước):")
    show([["·" if ch == "0" else ch for ch in row] for row in g])
    return label

_ = trace_islands(["11000", "11011", "00100", "10011"])

phát hiện đảo A tại (0, 0)
phát hiện đảo B tại (1, 3)
phát hiện đảo C tại (2, 2)
phát hiện đảo D tại (3, 0)
phát hiện đảo E tại (3, 3)
lưới sau khi gán nhãn (· là nước):
    AA···
    AA·BB
    ··C··
    D··EE


# 7. Phiên bản đúng format LeetCode

Dùng BFS để an toàn với lưới `300 × 300` (DFS đệ quy có thể sâu tới 90 000 tầng).

In [19]:
class Solution:
    def numIslands(self, grid: List[List[str]]) -> int:
        m, n = len(grid), len(grid[0])
        count = 0
        for r in range(m):
            for c in range(n):
                if grid[r][c] != "1":
                    continue
                count += 1
                grid[r][c] = "0"
                q = deque([(r, c)])
                while q:
                    x, y = q.popleft()
                    for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == "1":
                            grid[nx][ny] = "0"
                            q.append((nx, ny))
        return count

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.numIslands(to_grid(G1)), 1)
check("Ví dụ 2", sol.numIslands(to_grid(G2)), 3)
check("Toàn nước", sol.numIslands(to_grid(["000"])), 0)
check("Chéo không nối", sol.numIslands(to_grid(["10", "01"])), 2)
check("Một ô", sol.numIslands(to_grid(["1"])), 1)


def count_brute(rows):
    """Đếm bằng cách gán nhãn lặp đi lặp lại (không dùng DFS/BFS) — chậm nhưng độc lập."""
    m, n = len(rows), len(rows[0])
    lab = [[(r * n + c) if rows[r][c] == "1" else -1 for c in range(n)] for r in range(m)]
    changed = True
    while changed:
        changed = False
        for r in range(m):
            for c in range(n):
                if lab[r][c] < 0:
                    continue
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and lab[nr][nc] >= 0 and lab[nr][nc] < lab[r][c]:
                        lab[r][c] = lab[nr][nc]; changed = True
    return len({v for row in lab for v in row if v >= 0})


random.seed(200)
for _ in range(1000):
    m, n = random.randint(1, 7), random.randint(1, 7)
    rows = ["".join(random.choice("01") for _ in range(n)) for _ in range(m)]
    exp = count_brute(rows)
    assert sol.numIslands(to_grid(rows)) == exp == num_islands_dfs(to_grid(rows)) == num_islands_dsu(to_grid(rows)), rows
print("1000 lưới ngẫu nhiên: BFS, DFS, Union-Find khớp cách gán nhãn lặp")

big = ["".join(random.choice("0111") for _ in range(300)) for _ in range(300)]
for name, f in [("BFS", sol.numIslands), ("Union-Find", num_islands_dsu)]:
    t0 = time.time(); r = f(to_grid(big)); t1 = time.time()
    print(f"300 × 300, {name:<10}: {t1 - t0:.3f}s, {r} đảo")

Ví dụ 1        PASS  1


Ví dụ 2        PASS  3
Toàn nước      PASS  0
Chéo không nối PASS  2
Một ô          PASS  1


1000 lưới ngẫu nhiên: BFS, DFS, Union-Find khớp cách gán nhãn lặp


300 × 300, BFS       : 0.184s, 311 đảo


300 × 300, Union-Find: 0.241s, 311 đảo


# 8. Vì sao đúng? Độ phức tạp?

- Tô loang từ một ô đất đi qua đúng **toàn bộ** ô đất liên thông với nó (định nghĩa của thành phần liên thông) và đánh dấu chúng.
- Một ô đất chưa đánh dấu khi vòng quét gặp tới phải thuộc một đảo **chưa** được đếm (nếu đảo đó đã được đếm thì mọi ô của nó đã bị đánh dấu).
- Union-Find: mỗi `union` thành công gộp hai thành phần thành một; sau khi xét mọi cạnh kề, số tập = số thành phần liên thông.

```text
DFS/BFS:     Time O(m·n), Space O(m·n) (ngăn xếp/hàng đợi trường hợp xấu)
Union-Find:  Time O(m·n·α(m·n)) ≈ O(m·n), Space O(m·n)
```

(`α` là hàm Ackermann ngược — tăng cực chậm, `≤ 4` với mọi kích thước thực tế.)

# 9. Những lỗi dễ mắc

## Lỗi 1: DFS đệ quy trên lưới lớn → `RecursionError` (hoặc sập chương trình)

In [21]:
full40 = ["1" * 40 for _ in range(40)]           # một đảo 1600 ô
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)                       # giới hạn mặc định của Python
try:
    num_islands_dfs(to_grid(full40))
    print("DFS đệ quy chạy được")
except RecursionError:
    print("RecursionError: lưới 40 × 40 toàn đất đã cần độ sâu đệ quy > 1000")
finally:
    sys.setrecursionlimit(old_limit)
print("BFS trên lưới 300 × 300 toàn đất:", sol.numIslands(to_grid(["1" * 300 for _ in range(300)])), "đảo")

RecursionError: lưới 40 × 40 toàn đất đã cần độ sâu đệ quy > 1000


BFS trên lưới 300 × 300 toàn đất: 1 đảo


Độ sâu đệ quy có thể bằng số ô của đảo: lưới `300 × 300` toàn đất cần tới 90 000 tầng. Nếu tăng giới hạn bằng `sys.setrecursionlimit` quá lớn, chương trình có thể **sập hẳn** (tràn ngăn xếp của C) thay vì báo lỗi. Dùng BFS hoặc DFS bằng ngăn xếp tự quản lý.

## Lỗi 2: Đánh dấu khi **lấy ra** khỏi hàng đợi thay vì khi **đưa vào**

In [22]:
def bfs_mark_on_pop(rows):
    g = to_grid(rows); m, n = len(g), len(g[0])
    count, pushes = 0, 0
    for r in range(m):
        for c in range(n):
            if g[r][c] != "1":
                continue
            count += 1
            q = deque([(r, c)])
            while q:
                x, y = q.popleft()
                if g[x][y] != "1":
                    continue
                g[x][y] = "0"                     # đánh dấu muộn
                for dr, dc in DIRS:
                    nx, ny = x + dr, y + dc
                    if 0 <= nx < m and 0 <= ny < n and g[nx][ny] == "1":
                        q.append((nx, ny)); pushes += 1
    return count, pushes

def bfs_mark_on_push(rows):
    g = to_grid(rows); m, n = len(g), len(g[0])
    pushes = 0
    for r in range(m):
        for c in range(n):
            if g[r][c] != "1":
                continue
            g[r][c] = "0"; q = deque([(r, c)])
            while q:
                x, y = q.popleft()
                for dr, dc in DIRS:
                    nx, ny = x + dr, y + dc
                    if 0 <= nx < m and 0 <= ny < n and g[nx][ny] == "1":
                        g[nx][ny] = "0"; q.append((nx, ny)); pushes += 1
    return pushes

full = ["1" * 50 for _ in range(50)]
print("đánh dấu khi lấy ra : (số đảo, số lần đẩy vào hàng đợi) =", bfs_mark_on_pop(full))
print("đánh dấu khi đưa vào: số lần đẩy =", bfs_mark_on_push(full))

đánh dấu khi lấy ra : (số đảo, số lần đẩy vào hàng đợi) = (1, 4900)
đánh dấu khi đưa vào: số lần đẩy = 2499


Kết quả vẫn đúng (vì có kiểm tra khi lấy ra) nhưng mỗi ô bị đẩy vào hàng đợi **nhiều lần** (tối đa 4) → tốn gấp đôi, gấp ba. Không có dòng kiểm tra `if g[x][y] != "1": continue` thì còn **đếm sai** / lặp vô ích.

## Lỗi 3: So với số thay vì chuỗi

In [23]:
def wrong_int_compare(rows):
    g = to_grid(rows)
    return sum(1 for row in g for v in row if v == 1)     # SAI: '1' != 1

print("Sai :", wrong_int_compare(G2), "ô đất | thực tế:", sum(row.count("1") for row in G2))

Sai : 0 ô đất | thực tế: 7


## Lỗi 4: Tính cả ô **chéo** là kề

Đề chỉ tính 4 hướng; thêm 4 hướng chéo sẽ gộp nhầm `["10", "01"]` thành một đảo.

## Lỗi 5: Đánh dấu bằng một giá trị **trùng** với ký hiệu đất

Khi gán nhãn đảo bằng số thứ tự (`str(label)`), đảo đầu tiên có nhãn `"1"` — trùng ký hiệu đất! Các ô đã thăm vẫn "trông như" chưa thăm, BFS đẩy chúng vào hàng đợi mãi mãi → lặp vô hạn, tràn bộ nhớ. (Lỗi này xảy ra thật khi soạn hàm `trace_islands` ở mục 6 — vì vậy ở đó dùng nhãn chữ `A, B, C…`.) Ký hiệu "đã thăm" phải **khác** mọi giá trị "chưa thăm".

# 10. Mẹo nhận diện pattern

- **Lưới + "vùng liên thông"** → tô loang (DFS/BFS) từ mỗi ô chưa thăm; đếm số lần bắt đầu.
- **Kết nối thay đổi dần (thêm cạnh/đất theo thời gian)** → Union-Find (bài tập 5).
- **Lưới lớn** → BFS hoặc DFS bằng ngăn xếp, tránh đệ quy sâu.

Bài liên quan: **#695 Max Area of Island**, **#130 Surrounded Regions**, **#463 Island Perimeter** (Easy), **#694 Number of Distinct Islands**, **#305 Number of Islands II** (Hard), **#547 Number of Provinces**, **#1254 Number of Closed Islands**.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Đếm đảo và gán nhãn cho lưới `["10101", "11001", "00111"]`.

## Bài 2 — Điền chỗ trống (BFS)

In [24]:
def exercise_2_num_islands(grid: List[List[str]]) -> int:
    m, n = len(grid), len(grid[0])
    count = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] != "1":
                continue
            count += 1
            grid[r][c] = "0"
            q = deque([(r, c)])
            # TODO: BFS loang và đánh dấu
            pass
    return count


# assert exercise_2_num_islands(to_grid(["11110", "11010", "11000", "00000"])) == 1
# assert exercise_2_num_islands(to_grid(["11000", "11000", "00100", "00011"])) == 3
# assert exercise_2_num_islands(to_grid(["10", "01"])) == 2

## Bài 3 — Biến thể: Max Area of Island (LeetCode #695)

Lưới số nguyên `0/1`; trả về **diện tích** (số ô) của đảo lớn nhất, `0` nếu không có.

In [25]:
def exercise_3_max_area(grid: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_3_max_area([[0, 0, 1, 0, 0], [0, 1, 1, 0, 1], [0, 0, 0, 1, 1]]) == 3
# assert exercise_3_max_area([[0, 0, 0]]) == 0

## Bài 4 — Biến thể: không được sửa lưới

Đếm đảo bằng `set` các ô đã thăm; sau khi chạy, `grid` phải **giữ nguyên**.

In [26]:
def exercise_4_count_no_modify(grid: List[List[str]]) -> int:
    # TODO
    pass


# g4 = to_grid(["11000", "11000", "00100", "00011"]); assert exercise_4_count_no_modify(g4) == 3 and g4 == to_grid(["11000", "11000", "00100", "00011"])

## Bài 5 — Mini challenge: Number of Islands II (LeetCode #305, Hard)

Lưới `m × n` ban đầu toàn nước. Lần lượt biến các ô `positions[i]` thành đất; sau **mỗi** thao tác, trả về số đảo hiện có. Dùng Union-Find: thêm ô mới = thêm một tập (`+1`), rồi `union` với các ô đất kề (mỗi lần thành công `−1`). Một ô có thể được thêm hai lần (lần sau không đổi gì).

In [27]:
def exercise_5_num_islands2(m: int, n: int, positions: List[List[int]]) -> List[int]:
    # TODO
    pass


# assert exercise_5_num_islands2(3, 3, [[0, 0], [0, 1], [1, 2], [2, 1]]) == [1, 1, 2, 3]
# assert exercise_5_num_islands2(1, 1, [[0, 0]]) == [1]
# assert exercise_5_num_islands2(2, 2, [[0, 0], [1, 1], [0, 1], [0, 1]]) == [1, 2, 1, 1]

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_islands(["10101", "11001", "00111"])` sau khi tự làm.

**Bài 3**
- Mức 1: như BFS gốc, đếm số ô đã lấy ra trong mỗi lượt loang; giữ `max`.

**Bài 4**
- Mức 1: `seen = set()`; đánh dấu bằng `seen.add((r, c))` thay vì sửa `grid`.

**Bài 5**
- Mức 1: `DSU(m * n)` với `components = 0`; mảng `land` đánh dấu ô đã là đất.
- Mức 2: thêm `(r, c)`: nếu đã là đất → ghi lại số hiện tại; ngược lại `components += 1`, `union` với 4 ô kề đã là đất.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [28]:
print("→", trace_islands(["10101", "11001", "00111"]), "đảo")

phát hiện đảo A tại (0, 0)
phát hiện đảo B tại (0, 2)
phát hiện đảo C tại (0, 4)
lưới sau khi gán nhãn (· là nước):
    A·B·C
    AA··C
    ··CCC
→ 3 đảo


## Đáp án Bài 2

In [29]:
def answer_2_num_islands(grid: List[List[str]]) -> int:
    return Solution().numIslands(grid)

assert answer_2_num_islands(to_grid(["11110", "11010", "11000", "00000"])) == 1
assert answer_2_num_islands(to_grid(["11000", "11000", "00100", "00011"])) == 3
assert answer_2_num_islands(to_grid(["10", "01"])) == 2
print("Bài 2 PASS (giống Solution ở mục 7)")

Bài 2 PASS (giống Solution ở mục 7)


## Đáp án Bài 3

In [30]:
def answer_3_max_area(grid: List[List[int]]) -> int:
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    best = 0
    for r in range(m):
        for c in range(n):
            if g[r][c] != 1:
                continue
            g[r][c] = 0
            q, area = deque([(r, c)]), 0
            while q:
                x, y = q.popleft(); area += 1
                for dr, dc in DIRS:
                    nx, ny = x + dr, y + dc
                    if 0 <= nx < m and 0 <= ny < n and g[nx][ny] == 1:
                        g[nx][ny] = 0; q.append((nx, ny))
            best = max(best, area)
    return best

assert answer_3_max_area([[0, 0, 1, 0, 0], [0, 1, 1, 0, 1], [0, 0, 0, 1, 1]]) == 3
assert answer_3_max_area([[0, 0, 0]]) == 0
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [31]:
def answer_4_count_no_modify(grid: List[List[str]]) -> int:
    m, n = len(grid), len(grid[0])
    seen, count = set(), 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] != "1" or (r, c) in seen:
                continue
            count += 1
            seen.add((r, c)); q = deque([(r, c)])
            while q:
                x, y = q.popleft()
                for dr, dc in DIRS:
                    nx, ny = x + dr, y + dc
                    if 0 <= nx < m and 0 <= ny < n and grid[nx][ny] == "1" and (nx, ny) not in seen:
                        seen.add((nx, ny)); q.append((nx, ny))
    return count

g4 = to_grid(["11000", "11000", "00100", "00011"]); assert answer_4_count_no_modify(g4) == 3 and g4 == to_grid(["11000", "11000", "00100", "00011"])
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [32]:
def answer_5_num_islands2(m: int, n: int, positions: List[List[int]]) -> List[int]:
    dsu = DSU(m * n)
    dsu.components = 0
    land = [[False] * n for _ in range(m)]
    out = []
    for r, c in positions:
        if not land[r][c]:
            land[r][c] = True
            dsu.components += 1
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and land[nr][nc]:
                    dsu.union(r * n + c, nr * n + nc)
        out.append(dsu.components)
    return out

assert answer_5_num_islands2(3, 3, [[0, 0], [0, 1], [1, 2], [2, 1]]) == [1, 1, 2, 3]
assert answer_5_num_islands2(1, 1, [[0, 0]]) == [1]
assert answer_5_num_islands2(2, 2, [[0, 0], [1, 1], [0, 1], [0, 1]]) == [1, 2, 1, 1]
for _ in range(300):
    m, n = random.randint(1, 5), random.randint(1, 5)
    pos = [[random.randrange(m), random.randrange(n)] for _ in range(random.randint(1, 10))]
    got = answer_5_num_islands2(m, n, pos)
    grid = [["0"] * n for _ in range(m)]
    for (r, c), k in zip(pos, got):
        grid[r][c] = "1"
        assert k == count_brute(["".join(row) for row in grid])
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Quét lưới; gặp ô `'1'` chưa thăm thì `count += 1` và **tô loang** (BFS/DFS) đánh dấu cả đảo — đánh dấu ngay khi đưa vào hàng đợi. Union-Find cho cùng kết quả và mạnh khi đất được thêm dần. Dùng BFS hoặc ngăn xếp cho lưới lớn để tránh `RecursionError`.

---

## Checklist trước khi nộp

- [ ] So với chuỗi `'1'` (không phải số)?
- [ ] Chỉ 4 hướng, kiểm tra biên?
- [ ] Đánh dấu ngay khi đưa vào hàng đợi / ngay khi vào DFS?
- [ ] Không đệ quy quá sâu trên lưới 300 × 300?